# TorlakTag — Batch Inference: EXB → CoNLL-U

Runs the trained **XLM-RoBERTa-large** checkpoint over every `.exb` file in the
input directory and writes one `.conllu` file per source file.

**Resumable:** already-completed files are skipped automatically on re-run.

| Setting | Value |
|---|---|
| Model | `FacebookAI/xlm-roberta-large` (best checkpoint) |
| Input | `/content/drive/MyDrive/TorlakData/exb/` |
| Output | `/content/drive/MyDrive/TorlakData/final_conllu/` |
| Heads | LEMMA · UPOS · FEATS · XPOS |

## 0 · Setup

In [ ]:
!pip -q install lxml

import re, json, time
from pathlib import Path
from typing import Dict, List, Tuple
from dataclasses import dataclass
from collections import Counter

import pandas as pd
from lxml import etree

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig

from google.colab import drive
drive.mount("/content/drive")

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1 · Configuration

The only cell you may need to edit.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
MODEL_DIR     = Path("/content/drive/MyDrive/TorlakTag/models/multirun_20260213_064842/FacebookAI_xlm-roberta-large")
SPK_META_PATH = Path("/content/drive/MyDrive/TorlakData/spk.metadata.txt")
EXB_DIR       = Path("/content/drive/MyDrive/TorlakData/exb")
OUT_DIR       = Path("/content/drive/MyDrive/TorlakData/final_conllu")

# ── Model settings (must match training) ───────────────────────────────────
MODEL_HUB_NAME = "FacebookAI/xlm-roberta-large"
MAX_LEN        = 16
USE_FP16       = True
INFER_BATCH    = 256   # tokens per forward pass; lower if you hit OOM

# Optional: geo lookup table (place → lat/lon).  Set to None to skip.
GEO_TSV = None   # e.g. Path("/content/drive/MyDrive/TorlakData/place_geo.tsv")

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("✅ config ready")
print(f"   EXB input : {EXB_DIR}")
print(f"   CoNLL-U output: {OUT_DIR}")
print(f"   Model checkpoint: {MODEL_DIR}")

## 2 · Token normalisation

In [ ]:
def apply_spec_mapping(s: str) -> str:
    if s is None:
        return ""
    s = s.replace("#", "")
    s = s.replace("W", "Ə").replace("w", "ə")
    s = s.replace("1", "ḱ").replace("6", "ḱ")
    s = s.replace("2", "ǵ")
    s = s.replace("3", "č")
    s = s.replace("x", "š").replace("X", "š")
    s = s.replace("5", "ƨ")
    s = s.replace("ššš", "XXX")
    return s

RE_OVERLAP   = re.compile(r"\[[^\]]*\]")
RE_DOUBLEPAR = re.compile(r"^\(\(.*\)\)$")
RE_BULLETS   = re.compile(r"^[•]+$")
RE_LONGVOWEL = re.compile(r"([aeiouə])\1+")
RE_SPACES    = re.compile(r"\s+")
_STRIP_EDGE  = " \t\r\n\"'\u201c\u201d\u201e`´.,;:!?(){}[]<>•"

def is_x_special_token(tok: str) -> bool:
    t = tok.strip()
    if not t:
        return True
    if RE_DOUBLEPAR.match(t):
        return True
    if RE_BULLETS.match(t):
        return True
    return False

def strip_attached_specials(tok: str) -> str:
    t = tok.strip()
    t = re.sub(r"/+$", "", t)
    t = t.strip(_STRIP_EDGE)
    return t

def normalize_word(tok: str) -> str:
    t = apply_spec_mapping(tok).lower()
    t = RE_LONGVOWEL.sub(r"\1", t)
    t = RE_SPACES.sub(" ", t).strip()
    return t

def tokenize_with_rules(raw_text: str):
    if raw_text is None:
        return [], []
    s = apply_spec_mapping(raw_text).lower()
    s = RE_OVERLAP.sub(" ", s)
    raw_tokens = [t for t in s.split() if t.strip()]
    tokens, special = [], []
    for rt in raw_tokens:
        if is_x_special_token(rt):
            tokens.append(rt)
            special.append(True)
            continue
        has_alnum = any(ch.isalpha() or ch.isdigit() for ch in rt)
        if has_alnum:
            w = strip_attached_specials(rt)
            w = normalize_word(w)
            if w:
                tokens.append(w)
                special.append(False)
            else:
                tokens.append(rt)
                special.append(True)
        else:
            tokens.append(rt)
            special.append(True)
    return tokens, special

print("✅ normalisation ready")

## 3 · Speaker metadata

In [ ]:
df_spk = pd.read_csv(SPK_META_PATH, sep="\t", dtype=str, keep_default_na=False)
df_spk.columns = [c.strip() for c in df_spk.columns]
for c in df_spk.columns:
    df_spk[c] = df_spk[c].astype(str).str.strip()

def _pick_col(cols, keywords):
    for kw in keywords:
        for c in cols:
            if kw in c.lower():
                return c
    return None

COL_NAME = _pick_col(df_spk.columns, ["name"])
COL_ID   = _pick_col(df_spk.columns, ["id"])
COL_LOC  = _pick_col(df_spk.columns, ["location", "place", "loc"])
COL_AGE  = _pick_col(df_spk.columns, ["age", "years"])
COL_GEN  = _pick_col(df_spk.columns, ["gen", "sex"])
COL_EDU  = _pick_col(df_spk.columns, ["education", "edu", "school"])

abbr2id, abbr2loc, abbr2age, abbr2gen, abbr2edu = {}, {}, {}, {}, {}
for _, r in df_spk.iterrows():
    abbr = r.get(COL_NAME, "").strip() if COL_NAME else ""
    if not abbr:
        continue
    if COL_ID  and r.get(COL_ID,  "").strip(): abbr2id [abbr] = r[COL_ID ].strip()
    if COL_LOC and r.get(COL_LOC, "").strip(): abbr2loc[abbr] = r[COL_LOC].strip()
    if COL_AGE and r.get(COL_AGE, "").strip(): abbr2age[abbr] = r[COL_AGE].strip()
    if COL_GEN and r.get(COL_GEN, "").strip(): abbr2gen[abbr] = r[COL_GEN].strip()
    if COL_EDU and r.get(COL_EDU, "").strip(): abbr2edu[abbr] = r[COL_EDU].strip()

GEO = {}
if GEO_TSV is not None and Path(GEO_TSV).exists():
    df_geo = pd.read_csv(GEO_TSV, sep="\t", dtype=str)
    for _, r in df_geo.iterrows():
        GEO[str(r["place"]).strip()] = (float(r["lat"]), float(r["lon"]))

def _key(abbr):
    if abbr in abbr2id or abbr in abbr2loc or abbr in abbr2age or abbr in abbr2gen or abbr in abbr2edu:
        return abbr
    return abbr.split("_")[0]

def speaker_id(abbr):     return abbr2id .get(_key(abbr), abbr)
def speaker_loc(abbr):    return abbr2loc.get(_key(abbr), "")
def speaker_age(abbr):    return abbr2age.get(_key(abbr), "_")
def speaker_gender(abbr): return abbr2gen.get(_key(abbr), "_")
def speaker_edu(abbr):
    edu = abbr2edu.get(_key(abbr), "").strip()
    return edu if edu else "no education"
def speaker_geo(abbr):
    loc = speaker_loc(abbr)
    return GEO.get(loc) if loc else None

print(f"✅ speaker metadata: {len(abbr2id)} IDs | {len(abbr2age)} ages | {len(abbr2gen)} genders | {len(abbr2edu)} edu")

## 4 · Model definition

In [ ]:
def get_hidden_size(cfg):
    if hasattr(cfg, "hidden_size"): return int(cfg.hidden_size)
    if hasattr(cfg, "d_model"):     return int(cfg.d_model)
    raise ValueError("Cannot infer hidden size from config")

def mean_pool(last_hidden, attention_mask):
    mask   = attention_mask.unsqueeze(-1).type_as(last_hidden)
    summed = (last_hidden * mask).sum(dim=1)
    return summed / mask.sum(dim=1).clamp(min=1.0)

class MultiHeadTokenTagger(nn.Module):
    def __init__(self, encoder_name, n_lemma, n_upos, n_feats, n_xpos, dropout=0.1):
        super().__init__()
        cfg          = AutoConfig.from_pretrained(encoder_name)
        self.encoder = AutoModel.from_pretrained(encoder_name, torch_dtype=torch.float32)
        h            = get_hidden_size(cfg)
        self.drop       = nn.Dropout(dropout)
        self.lemma_head = nn.Linear(h, n_lemma)
        self.upos_head  = nn.Linear(h, n_upos)
        self.feat_head  = nn.Linear(h, n_feats)
        self.xpos_head  = nn.Linear(h, n_xpos)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        x   = self.drop(mean_pool(out.last_hidden_state, attention_mask))
        return {
            "lemma_logits": self.lemma_head(x),
            "upos_logits":  self.upos_head(x),
            "feat_logits":  self.feat_head(x),
            "xpos_logits":  self.xpos_head(x),
        }

print("✅ model class defined")

## 5 · Load checkpoint + label maps

In [ ]:
# ── Label maps ─────────────────────────────────────────────────────────────
lemma2id = json.load(open(MODEL_DIR / "lemma2id.json",   encoding="utf-8"))
upos2id  = json.load(open(MODEL_DIR / "upos2id.json",    encoding="utf-8"))
feat2id  = json.load(open(MODEL_DIR / "feature2id.json", encoding="utf-8"))
xpos2id  = json.load(open(MODEL_DIR / "xpos2id.json",    encoding="utf-8"))

id2lemma = {int(v): k for k, v in lemma2id.items()}
id2upos  = {int(v): k for k, v in upos2id.items()}
id2feat  = {int(v): k for k, v in feat2id.items()}
id2xpos  = {int(v): k for k, v in xpos2id.items()}

print(f"Label vocab sizes — lemma: {len(lemma2id)} | upos: {len(upos2id)} | feats: {len(feat2id)} | xpos: {len(xpos2id)}")

# ── Tokenizer ──────────────────────────────────────────────────────────────
# Loaded from the model dir (saved there during training)
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), use_fast=True)

# ── Model ──────────────────────────────────────────────────────────────────
model = MultiHeadTokenTagger(
    MODEL_HUB_NAME,
    n_lemma=len(lemma2id),
    n_upos=len(upos2id),
    n_feats=len(feat2id),
    n_xpos=len(xpos2id),
    dropout=0.1,
).to(device)

ckpt = MODEL_DIR / "best_model.pt"
model.load_state_dict(torch.load(ckpt, map_location=device))
model.eval()

amp_dtype = torch.float16 if (USE_FP16 and device.type == "cuda") else None
print(f"✅ checkpoint loaded from {ckpt}")
print(f"   amp_dtype: {amp_dtype}")

## 6 · Inference helpers

In [ ]:
@torch.no_grad()
def predict_batch(tokens: List[str]) -> List[Tuple[str, str, str, str]]:
    """Return list of (lemma, upos, xpos, feats) for each token."""
    enc       = tokenizer(tokens, padding=True, truncation=True,
                          max_length=MAX_LEN, return_tensors="pt")
    input_ids = enc["input_ids"].to(device)
    attn      = enc["attention_mask"].to(device)

    if amp_dtype is not None:
        with torch.amp.autocast(device_type="cuda", dtype=amp_dtype):
            out = model(input_ids, attn)
    else:
        out = model(input_ids, attn)

    lem_ids  = out["lemma_logits"].argmax(dim=1).tolist()
    upos_ids = out["upos_logits"].argmax(dim=1).tolist()
    feat_ids = out["feat_logits"].argmax(dim=1).tolist()
    xpos_ids = out["xpos_logits"].argmax(dim=1).tolist()

    preds = []
    for tok, li, ui, fi, xi in zip(tokens, lem_ids, upos_ids, feat_ids, xpos_ids):
        lemma = id2lemma.get(li, "<UNK>")
        upos  = id2upos.get(ui, "X")
        feats = id2feat.get(fi, "_")
        xpos  = id2xpos.get(xi, "X")
        if lemma == "<UNK>": lemma = tok
        if feats == "<UNK>": feats = "_"
        if xpos  == "<UNK>": xpos  = "X"
        preds.append((lemma, upos, xpos, feats))
    return preds


print("✅ predict_batch ready")

# Quick sanity check
_test = predict_batch(["idem", "kući", "da"])
print("Sanity check — idem / kući / da:")
for tok, (lem, up, xp, fe) in zip(["idem", "kući", "da"], _test):
    print(f"  {tok:<10} lemma={lem:<12} upos={up:<6} xpos={xp:<10} feats={fe}")

## 7 · EXB parser + CoNLL-U writer

In [ ]:
GLOBAL_COLUMNS = "ID FORM LEMMA UPOS XPOS FEATS HEAD DEPREL DEPS MISC"


def parse_exb_utterances(exb_path: Path) -> List[Dict]:
    """Parse an EXMARaLDA .exb file and return a time-sorted list of utterances."""
    tree = etree.parse(str(exb_path))
    root = tree.getroot()

    timeline = root.find(".//common-timeline")
    tli_time: Dict[str, float] = {}
    for tli in timeline.iter("tli"):
        tid = tli.attrib.get("id")
        if tid and "time" in tli.attrib:
            try:
                tli_time[tid] = float(tli.attrib["time"])
            except ValueError:
                pass

    utterances = []
    file_id = exb_path.stem

    for tier in root.iter("tier"):
        display = tier.attrib.get("display-name", "")
        if "_" not in display:   # speaker tiers always have underscore
            continue
        speaker_abbr = display

        for ev in tier.iter("event"):
            if ev.text is None:
                continue
            start = ev.attrib.get("start")
            end   = ev.attrib.get("end")
            if start not in tli_time or end not in tli_time:
                continue
            utterances.append({
                "file_id":      file_id,
                "speaker_abbr": speaker_abbr,
                "start_time":   tli_time[start],
                "end_time":     tli_time[end],
                "raw_text":     ev.text,
            })

    utterances.sort(key=lambda x: x["start_time"])
    return utterances


def write_conllu_sentence(f, sent_id, tokens, preds, speaker_abbr, start_time, end_time):
    """Write one CoNLL-U sentence block with full metadata comments."""
    loc = speaker_loc(speaker_abbr)
    geo = speaker_geo(speaker_abbr)

    f.write(f"# sent_id = {sent_id}\n")
    f.write(f"# text = {' '.join(tokens)}\n")
    f.write(f"# speaker = {speaker_id(speaker_abbr)}\n")
    f.write(f"# speaker_abbr = {speaker_abbr}\n")
    if loc:
        f.write(f"# location = {loc}\n")
    f.write(f"# speaker_age = {speaker_age(speaker_abbr)}\n")
    f.write(f"# speaker_gender = {speaker_gender(speaker_abbr)}\n")
    f.write(f"# speaker_education = {speaker_edu(speaker_abbr)}\n")
    f.write(f"# start_time = {start_time:.3f}\n")
    f.write(f"# end_time = {end_time:.3f}\n")
    if geo is not None:
        f.write(f"# geo = {geo[0]:.6f},{geo[1]:.6f}\n")

    for i, (tok, (lem, up, xp, fe)) in enumerate(zip(tokens, preds), start=1):
        misc = f"MulText={xp}" if xp and xp not in ("_", "X") else "_"
        f.write(f"{i}\t{tok}\t{lem}\t{up}\t{xp}\t{fe}\t_\t_\t_\t{misc}\n")
    f.write("\n")


def export_exb_to_conllu(exb_path: Path, out_path: Path) -> int:
    """
    Convert one EXB file to CoNLL-U.  Returns the number of sentences written.
    Utterances with no tokens are silently skipped.
    """
    utterances = parse_exb_utterances(exb_path)
    sent_no = 0

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8") as f:
        f.write(f"# global.columns = {GLOBAL_COLUMNS}\n\n")

        for u in utterances:
            tokens, special_mask = tokenize_with_rules(u["raw_text"])
            if not tokens:
                continue

            # Run inference only on non-special tokens
            normal_tokens = [t for t, sp in zip(tokens, special_mask) if not sp]
            normal_preds  = []
            for i in range(0, len(normal_tokens), INFER_BATCH):
                normal_preds.extend(predict_batch(normal_tokens[i : i + INFER_BATCH]))

            # Re-merge: special tokens get a fixed X annotation
            preds = []
            j = 0
            for tok, sp in zip(tokens, special_mask):
                if sp:
                    preds.append((tok, "X", "X", "_"))
                else:
                    preds.append(normal_preds[j])
                    j += 1

            sent_no += 1
            sent_id  = f"{u['file_id']}-s{sent_no:04d}"
            write_conllu_sentence(
                f, sent_id, tokens, preds,
                u["speaker_abbr"], u["start_time"], u["end_time"]
            )

    return sent_no


print("✅ EXB parser and CoNLL-U writer ready")

## 8 · Batch inference — all EXB files

**Resumable:** if `<stem>.conllu` already exists in the output directory the file is skipped.
Re-run this cell after a disconnect to continue from where it stopped.

In [ ]:
exb_files = sorted(EXB_DIR.glob("*.exb"))
print(f"Found {len(exb_files)} EXB files in {EXB_DIR}\n")

total_sentences = 0
skipped = 0
failed  = []
t_start = time.time()

for idx, exb_path in enumerate(exb_files, start=1):
    out_path = OUT_DIR / (exb_path.stem + ".conllu")

    if out_path.exists():
        skipped += 1
        print(f"[{idx:>3}/{len(exb_files)}] SKIP (exists) {exb_path.name}")
        continue

    t0 = time.time()
    try:
        n_sents = export_exb_to_conllu(exb_path, out_path)
        elapsed = time.time() - t0
        total_sentences += n_sents
        print(f"[{idx:>3}/{len(exb_files)}] ✅ {exb_path.name:<35}  "
              f"{n_sents:>4} sentences  {elapsed:.1f}s")
    except Exception as e:
        elapsed = time.time() - t0
        failed.append(exb_path.name)
        print(f"[{idx:>3}/{len(exb_files)}] ❌ {exb_path.name}  ERROR: {e}  ({elapsed:.1f}s)")

total_elapsed = time.time() - t_start
done = len(exb_files) - skipped - len(failed)

print("\n" + "=" * 60)
print(f"Done.  {done} converted | {skipped} skipped | {len(failed)} failed")
print(f"Total sentences written this run: {total_sentences}")
print(f"Total time: {total_elapsed/60:.1f} min")
print(f"Output directory: {OUT_DIR}")

if failed:
    print("\nFailed files:")
    for f in failed:
        print(f"  {f}")

## 9 · Output summary

In [ ]:
conllu_files = sorted(OUT_DIR.glob("*.conllu"))
print(f"{len(conllu_files)} CoNLL-U files in {OUT_DIR}\n")

rows = []
for p in conllu_files:
    text = p.read_text(encoding="utf-8")
    n_sents = text.count("# sent_id")
    n_toks  = sum(1 for ln in text.splitlines()
                  if ln and not ln.startswith("#") and "\t" in ln)
    rows.append({"file": p.name, "sentences": n_sents, "tokens": n_toks})

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\nTotal — sentences: {df['sentences'].sum()} | tokens: {df['tokens'].sum()}")